# Fase 1: Gaming vs Academic Performance

**Integrantes:** José Zaavedra - Nicolás Arza

**Objetivo:** preparar un dataset limpio y reproducible para estudiar la asociación entre horas de videojuegos y rendimiento académico, considerando estudio y sueño. El dataset es simulado según su autor.

## Índice CRISP-DM

0. Configuración y carga — **COMPLETA**  
1. Comprensión del problema — **COMPLETA**  
2. Comprensión y selección de los datos — **COMPLETA**  
3. Limpieza y transformación — **COMPLETA**  
4. Análisis univariado y bivariado — **COMPLETA**  
5. Análisis descriptivo y exploratorio — **COMPLETA**  
6. Conclusión de la Fase 1 — **PENDIENTE**

## Sección 0. Configuración y carga
Se importan las librerías permitidas, se definen rutas relativas, se carga `df_raw` y se crea la copia de trabajo `df`.

In [ ]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd

print(f'pandas: {pd.__version__}')
print(f'numpy: {np.__version__}')

RAW_PATH = Path('../data/raw/Gaming_Academic_Performance_updated.csv')
CLEAN_PATH = Path('../data/processed/gaming_academic_clean.csv')
BITACORA_PATH = Path('../output/bitacora_limpieza.csv')
TOPE_NOTAS = 100

CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
BITACORA_PATH.parent.mkdir(parents=True, exist_ok=True)

df_raw = pd.read_csv(RAW_PATH)
n0 = len(df_raw)
hash_sha256 = hashlib.sha256(RAW_PATH.read_bytes()).hexdigest()

print(f'Dimensiones originales: {df_raw.shape}')
print('Tipos originales:')
print(df_raw.dtypes)
print(f'Hash SHA-256: {hash_sha256}')
print(f'n0: {n0}')

# Toda la limpieza se hace sobre esta copia; df_raw permanece sin modificar.
df = df_raw.copy()
registros_bitacora = []

def registrar(n_paso, paso, columna, revision_o_problema, registros_afectados, decision, fundamento):
    registros_bitacora.append({
        'n_paso': n_paso,
        'paso': paso,
        'columna': columna,
        'revision_o_problema': revision_o_problema,
        'registros_afectados': registros_afectados,
        'decision': decision,
        'fundamento': fundamento,
        'filas_despues': len(df)
    })

: 

## Sección 1. Comprensión del problema

### 1.1 Contexto

Los videojuegos comparten el tiempo diario con el estudio, el sueño y otras actividades digitales. La evidencia sobre su relación con el rendimiento académico no es uniforme: algunos estudios encuentran asociaciones pequeñas o nulas y otros observan asociaciones negativas, especialmente ante uso problemático. Por esta heterogeneidad, este proyecto estudia **asociaciones**, no causalidad [1-4].

El dataset contiene 8.000 estudiantes de 16 a 24 años. La unidad de análisis es el estudiante y `grades` es la variable objetivo, interpretada en escala 0-100 después de tratar los valores superiores a 100.

### 1.2 Pregunta de investigación

¿En qué medida las horas diarias dedicadas a videojuegos se asocian con el rendimiento académico, considerando las horas de estudio y de sueño?

### 1.3 Objetivo general

Analizar la asociación entre `gaming_hours` y el rendimiento académico, considerando `study_hours`, `sleep_hours` y `uso_no_gaming` como variables de contexto y ajuste.

### 1.4 Objetivos específicos

1. Caracterizar las horas de juego y sus niveles `Bajo`, `Medio` y `Alto`, junto con la distribución de `grades`.
2. Medir y comparar la asociación de `gaming_hours`, `study_hours` y `sleep_hours` con las calificaciones.
3. Evaluar si la asociación entre juego y rendimiento se mantiene al ajustar por estudio, sueño y uso no gaming, sin interpretarlo como causalidad.
4. Construir en la siguiente fase una regresión lineal con partición de entrenamiento y prueba, comparando una línea base mediante RMSE, MAE y R².
5. Explorar si la relación es aproximadamente lineal o cambia por tramos, usando terciles, cortes fijos de 2, 4 y 6 horas y una comprobación no paramétrica.

### 1.5 Justificación

La pregunta permite practicar un flujo completo de ciencia de datos y estudiar cómo distintas dimensiones del uso del tiempo se relacionan con el rendimiento académico. Como el dataset presenta indicios de simulación y no documenta un muestreo poblacional, los resultados son un ejercicio analítico y no evidencia suficiente para decisiones reales.

### 1.6 Alcance y limitaciones

- El diseño es transversal; no permite establecer causalidad ni dirección temporal.
- La recolección no está documentada y la posible simulación limita la generalización.
- `gaming_genre`, `reaction_time_ms` y `addiction_score` quedan fuera del análisis principal. `reaction_time_ms` presenta correlación cercana a `-0,94` con `gaming_hours` y `addiction_score` contiene valores negativos.
- `gender=Other` contiene 293 estudiantes, por lo que sus comparaciones tienen menor precisión.
- Se recortan 134 valores de `grades > 100` a 100. Como ya había 439 valores iguales a 100, esto produce un efecto techo que se considerará en el análisis de sensibilidad.
- `stress_level` se mantiene para descripción, pero se excluye del ajuste principal porque resulta casi determinística respecto de las horas de juego, estudio y sueño.

## Sección 2. Comprensión y selección de los datos

### 2.1 Inventario de archivos disponibles

La fuente es el dataset simulado **Gaming vs Academic Performance**, publicado en Kaggle por `aiexplorer77`. La entrada es `Gaming_Academic_Performance_updated.csv` y la salida limpia es `gaming_academic_clean.csv`. El corte y la verificación corresponden al 19/09/2026. Es un dataset transversal y no contiene variables temporales.

### 2.2 Diccionario de datos original

| Campo | Descripción | Tipo | Unidad, rango o categorías |
|---|---|---|---|
| `student_id` | Identificador único | Entero | 1 a 8.000 |
| `age` | Edad del estudiante | Entero | 16 a 24 años |
| `gender` | Género autorreportado | Categórica nominal | Male / Female / Other |
| `gaming_hours` | Horas diarias de videojuegos | Numérica continua | 0,00 a 8,00 horas |
| `study_hours` | Horas diarias de estudio | Numérica continua | 1,00 a 10,00 horas |
| `sleep_hours` | Horas diarias de sueño | Numérica continua | 4,00 a 9,00 horas |
| `attendance` | Asistencia a clases | Numérica continua | 60,00% a 100,00% |
| `gaming_genre` | Género principal de videojuego | Categórica nominal | FPS / RPG / Casual |
| `social_activity` | Nivel de actividad social | Numérica en escala | 0,00 a 5,00 |
| `device_usage` | Uso diario general de dispositivos | Numérica continua | 1,10 a 13,95 horas |
| `reaction_time_ms` | Tiempo de reacción | Numérica continua | 183,26 a 347,87 ms |
| `addiction_score` | Puntaje de adicción a videojuegos | Numérica continua | -4,51 a 23,16 |
| `stress_level` | Nivel de estrés autopercibido | Categórica ordinal | Low / Medium / High |
| `grades` | Rendimiento académico | Numérica continua | 0,00 a 118,63 |

No se observaron faltantes en las 14 columnas.

### 2.3 Unidad de análisis y clave primaria

Cada fila representa un estudiante. `student_id` es único en los 8.000 registros y se elimina del dataset analítico porque identifica el registro, pero no es una variable explicativa.

### 2.4 Perfilado inicial

El dataset original contiene 8.000 filas y 14 columnas, sin faltantes ni filas duplicadas. Las variables categóricas tienen tres categorías. Medias observadas: `gaming_hours` = 4,08; `study_hours` = 5,45; `sleep_hours` = 6,48; `grades` original = 66,18. La edad se distribuye casi uniformemente entre 16 y 24 años.

### 2.5 Evaluación de calidad

Hay 134 registros con `grades > 100` y 107 con `addiction_score < 0`. No hay faltantes ni duplicados. El máximo original de `grades` es 118,63. Se asume una escala de 0 a 100 para limitar `grades`; esta decisión puede producir un efecto techo.

### 2.6 Selección justificada de variables

Se conservan variables demográficas (`age`, `gender`), de contexto (`attendance`, `social_activity`, `stress_level`), de uso (`gaming_hours`, `study_hours`, `sleep_hours`, `device_usage`) y variables de trazabilidad o derivadas (`grades_original`, flags, `uso_no_gaming`, `gaming_cat`).

Para el análisis principal se usarán `gaming_hours`, `study_hours`, `sleep_hours` y `uso_no_gaming`. `device_usage` no debe entrar junto con ellas porque `device_usage = gaming_hours + uso_no_gaming`, lo que produce dependencia lineal exacta. `reaction_time_ms`, `gaming_genre`, `addiction_score`, `student_id` y `grades_original` se excluyen de los modelos principales por redundancia, alcance, calidad o trazabilidad.

### 2.7 Documentación de la fuente

Fuente: [Kaggle, Gaming vs Academic Performance](https://www.kaggle.com/datasets/aiexplorer77/gaming-vs-academic-performance), usuario `aiexplorer77`. La fecha de recolección no está documentada. No se asigna una licencia por inferencia ni se afirma una versión específica de la ficha. El dataset se describe como simulado según su autor.

Referencias externas de contexto: Drummond & Sauer (2014), Adelantado-Renau et al. (2019), Alzahrani & Griffiths (2024) y Chew, Yow & Dimech (2026).

## Sección 3. Limpieza y transformación
Se documentan las revisiones, se transforma una copia y no se eliminan filas ni se imputan valores.

### 3.1 Revisiones previas
Se revisan faltantes, duplicados, unicidad, categorías, rangos, atípicos y consistencias sin modificar los datos.

In [ ]:
# Faltantes por columna.
faltantes = df.isna().sum()
for columna, cantidad in faltantes.items():
    registrar('3.1', 'Revisiones previas', columna, 'Valores faltantes', int(cantidad), 'No imputar', 'Se conservan las filas y se documentan los faltantes.')

duplicadas_completas = int(df.duplicated().sum())
registrar('3.1', 'Revisiones previas', 'todas', 'Filas duplicadas completas', duplicadas_completas, 'No eliminar filas', 'La consigna exige conservar las 8000 filas.')
duplicadas_sin_id = int(df.drop(columns=['student_id']).duplicated().sum())
registrar('3.1', 'Revisiones previas', 'todas excepto student_id', 'Filas duplicadas ignorando student_id', duplicadas_sin_id, 'No eliminar filas', 'La revisión informa calidad sin alterar el dataset.')

id_notnull = int(df['student_id'].notna().sum())
id_nunique = int(df['student_id'].nunique())
id_unico = bool(df['student_id'].is_unique)
fallos_id = (n0 - id_notnull) + (n0 - id_nunique) + int(not id_unico)
registrar('3.1', 'Revisiones previas', 'student_id', 'Unicidad: notnull, nunique y is_unique', fallos_id, 'Conservar para revisar y descartar después', f'notnull={id_notnull}; nunique={id_nunique}; is_unique={id_unico}.')

nombres_duplicados = int(pd.Index(df.columns).duplicated().sum())
columnas_duplicadas = int(df.T.duplicated().sum())
registrar('3.1', 'Revisiones previas', 'nombres de columnas', 'Nombres de columna duplicados', nombres_duplicados, 'No modificar', 'Los nombres deben ser únicos.')
registrar('3.1', 'Revisiones previas', 'columnas', 'Columnas con contenido idéntico', columnas_duplicadas, 'No modificar', 'Se evita conservar información repetida sin cambiar el original.')

espacios_por_categoria = {}
variantes_por_categoria = {}
for columna in ['gender', 'gaming_genre', 'stress_level']:
    valores = df[columna].dropna().astype(str)
    con_espacios = int((valores != valores.str.strip()).sum())
    grupos_mayusculas = valores.groupby(valores.str.lower()).nunique()
    variantes = int((grupos_mayusculas > 1).sum())
    espacios_por_categoria[columna] = con_espacios
    variantes_por_categoria[columna] = variantes
    registrar('3.1', 'Revisiones previas', columna, 'Categorías, espacios sobrantes y variantes de mayúsculas', con_espacios + variantes, f'Valores únicos: {sorted(valores.unique().tolist())}', 'Se documentan las categorías antes de normalizar tipos.')
    print(f'{columna}: valores={sorted(valores.unique().tolist())}; espacios={con_espacios}; grupos_con_variantes_mayusculas={variantes}')

columnas_numericas = df.select_dtypes(include=np.number).columns
atipicos_por_columna = {}
for columna in columnas_numericas:
    minimo = df[columna].min()
    maximo = df[columna].max()
    q1 = df[columna].quantile(0.25)
    q3 = df[columna].quantile(0.75)
    iqr = q3 - q1
    atipicos = int(((df[columna] < q1 - 1.5 * iqr) | (df[columna] > q3 + 1.5 * iqr)).sum())
    atipicos_por_columna[columna] = atipicos
    registrar('3.1', 'Revisiones previas', columna, 'Rango y atípicos por regla 1.5*IQR', atipicos, f'mín={minimo}; máx={maximo}', 'Se informa el rango y no se eliminan atípicos.')
    print(f'{columna}: mínimo={minimo}; máximo={maximo}; atípicos={atipicos}')

consistencias = {
    'device_usage >= gaming_hours': int((df['device_usage'] < df['gaming_hours']).sum()),
    'gaming_hours + study_hours + sleep_hours <= 24': int((df[['gaming_hours', 'study_hours', 'sleep_hours']].sum(axis=1) > 24).sum()),
    'attendance fuera de [0, 100]': int((~df['attendance'].between(0, 100)).sum()),
    'grades negativas': int((df['grades'] < 0).sum())
}
for revision, cantidad in consistencias.items():
    registrar('3.1', 'Revisiones previas', 'consistencia', revision, cantidad, 'Revisar sin eliminar filas', 'Las reglas de consistencia se informan antes de transformar.')
print('Consistencias:', consistencias)

def comparar(nombre, observado, esperado, tolerancia=0):
    estado = 'OK' if abs(observado - esperado) <= tolerancia else 'REVISAR'
    print(f'{nombre}: observado={observado}; esperado={esperado}; {estado}')

comparar('faltantes totales', int(faltantes.sum()), 0)
comparar('duplicadas completas', duplicadas_completas, 0)
comparar('duplicadas ignorando student_id', duplicadas_sin_id, 0)
comparar('student_id distintos', id_nunique, n0)
comparar('student_id único', int(id_unico), 1)
comparar('nombres de columna duplicados', nombres_duplicados, 0)
comparar('columnas con contenido idéntico', columnas_duplicadas, 0)
for columna in espacios_por_categoria:
    comparar(f'{columna} con espacios sobrantes', espacios_por_categoria[columna], 0)
    comparar(f'{columna} con variantes de mayúsculas', variantes_por_categoria[columna], 0)
comparar('atípicos totales', sum(atipicos_por_columna.values()), 0)
comparar('device_usage menor que gaming_hours', consistencias['device_usage >= gaming_hours'], 0)

### 3.2 Indicador de registros ajustados
Se marca `grades` con más de dos decimales antes de redondear. La relación con los ajustes de horas es una hipótesis, no un hecho confirmado.

In [ ]:
flag_ajustado = (df['grades'] - df['grades'].round(2)).abs() > 1e-9
df['flag_ajustado'] = flag_ajustado.astype(bool)
cantidad_ajustados = int(flag_ajustado.sum())
suma_horas = df[['gaming_hours', 'study_hours', 'sleep_hours']].sum(axis=1)
promedio_ajustadas = suma_horas[flag_ajustado].mean()
promedio_no_ajustadas = suma_horas[~flag_ajustado].mean()
print(f'Registros con más de dos decimales en grades: {cantidad_ajustados}')
print(f'Media de horas totales en filas ajustadas: {promedio_ajustadas:.4f}')
print(f'Media de horas totales en filas no ajustadas: {promedio_no_ajustadas:.4f}')
registrar('3.2', 'Indicador de registros ajustados', 'grades', 'Más de dos decimales antes del redondeo', cantidad_ajustados, 'Crear flag_ajustado', 'La relación con ajustes de horas es una hipótesis y se conserva como indicador.')
comparar('grades con más de dos decimales', cantidad_ajustados, 1671, tolerancia=100)

### 3.3 Limpieza de `grades`
Se conserva `grades_original`, se marca `flag_grades_gt100` y se limita `grades` a 100 bajo el supuesto de una escala 0-100.

In [ ]:
grades_original = df['grades'].copy()
flag_grades_gt100 = df['grades'] > TOPE_NOTAS
cantidad_gt100 = int(flag_grades_gt100.sum())
cantidad_100 = int((df['grades'] == 100).sum())
cantidad_0 = int((df['grades'] == 0).sum())
print(f'Notas mayores que 100: {cantidad_gt100}')
print(f'Notas exactamente iguales a 100: {cantidad_100}')
print(f'Notas exactamente iguales a 0: {cantidad_0}')
print(f'Máximo original de grades: {grades_original.max()}')
comparar('máximo original de grades', float(grades_original.max()), 118.63, tolerancia=0.01)

df['grades_original'] = grades_original
df['flag_grades_gt100'] = flag_grades_gt100.astype(bool)
df['grades'] = df['grades'].clip(upper=TOPE_NOTAS)
registrar('3.3', 'Limpieza de grades', 'grades', 'Valores superiores al tope de notas', cantidad_gt100, 'Aplicar clip superior en 100 y conservar grades_original', 'Conserva las 8000 filas, sigue la escala 0-100 asumida y permite análisis de sensibilidad posterior.')
comparar('grades mayores que 100', cantidad_gt100, 134, tolerancia=20)
comparar('grades iguales a 100', cantidad_100, 439, tolerancia=20)
comparar('grades iguales a 0', cantidad_0, 1)

### 3.4 Tipos y categorías
Se quitan espacios externos y se convierten las categorías; `stress_level` queda ordenada como Low < Medium < High.

In [ ]:
for columna in ['gender', 'stress_level', 'gaming_genre']:
    antes = df[columna].copy()
    df[columna] = df[columna].str.strip()
    cambiaron = int((antes != df[columna]).sum())
    registrar('3.4', 'Tipos y categorías', columna, 'Espacios externos', cambiaron, 'Aplicar str.strip()', 'Normaliza categorías sin cambiar su significado.')
    print(f'{columna}: valores modificados por strip={cambiaron}')

df['gender'] = df['gender'].astype('category')
stress_categorias = ['Low', 'Medium', 'High']
df['stress_level'] = pd.Categorical(df['stress_level'], categories=stress_categorias, ordered=True)
age_entero = bool(np.all(df['age'].dropna() == df['age'].dropna().astype(int)))
registrar('3.4', 'Tipos y categorías', 'age', 'Verificación de tipo entero', 0 if age_entero else 1, 'Conservar tipo entero si se cumple', 'La edad está definida como años enteros.')
print(f'age es entero: {age_entero}')

### 3.5 Variables derivadas
Se supone que `device_usage` incluye el juego, por lo que se crea `uso_no_gaming`. Estas tres variables no deben entrar juntas en una regresión. `gaming_cat` usa terciles; cortes fijos 2 y 5 horas son una alternativa.

In [ ]:
device_menor = int((df['device_usage'] < df['gaming_hours']).sum())
print(f'Filas con device_usage < gaming_hours: {device_menor}')
registrar('3.5', 'Variables derivadas', 'uso_no_gaming', 'device_usage >= gaming_hours', device_menor, 'Crear diferencia solo si la consistencia se mantiene', 'Supuesto: device_usage incluye gaming_hours.')

df['uso_no_gaming'] = df['device_usage'] - df['gaming_hours']
df['gaming_cat'], cortes_gaming = pd.qcut(df['gaming_hours'], q=3, labels=['Bajo', 'Medio', 'Alto'], retbins=True)
df['gaming_cat'] = pd.Categorical(df['gaming_cat'], categories=['Bajo', 'Medio', 'Alto'], ordered=True)
tamanos_gaming = df['gaming_cat'].value_counts(sort=False)
print(f'Cortes de gaming_hours: {cortes_gaming}')
print('Tamaño de cada grupo:')
print(tamanos_gaming)
comparar('cortes de gaming_hours: tercil 1', float(cortes_gaming[1]), 2.8, tolerancia=0.1)
comparar('cortes de gaming_hours: tercil 2', float(cortes_gaming[2]), 5.46, tolerancia=0.1)
registrar('3.5', 'Variables derivadas', 'gaming_cat', 'Cortes por terciles', len(cortes_gaming) - 1, f'Cortes: {cortes_gaming.tolist()}', 'Agrupa en Bajo, Medio y Alto para comparación descriptiva; los cortes fijos 2 y 5 horas son una alternativa.')

### 3.6 Redondeo
Se redondean a dos decimales las columnas float, excepto `grades_original`.

In [ ]:
columnas_float = df.select_dtypes(include=['float64', 'float32']).columns.tolist()
columnas_float = [columna for columna in columnas_float if columna != 'grades_original']
for columna in columnas_float:
    df[columna] = df[columna].round(2)
excesos_decimales = int(sum(((df[columna] - df[columna].round(2)).abs() > 1e-9).sum() for columna in columnas_float))
comparar('valores float con más de dos decimales después del redondeo', excesos_decimales, 0)
registrar('3.6', 'Redondeo', ', '.join(columnas_float), 'Valores float con más de dos decimales', cantidad_ajustados, 'Redondear a dos decimales', 'Facilita lectura y mantiene grades_original sin alterar.')

### 3.7 Descarte de columnas
Se calcula la evidencia y se descartan identificador, variables redundantes o fuera del alcance.

In [ ]:
student_id_secuencial = bool((df_raw['student_id'] == np.arange(1, n0 + 1)).all())
corr_reaction = df_raw['gaming_hours'].corr(df_raw['reaction_time_ms'])
corr_addiction = df_raw['gaming_hours'].corr(df_raw['addiction_score'])
negativos_addiction = int((df_raw['addiction_score'] < 0).sum())
print(f'student_id es fila + 1: {student_id_secuencial}')
print(f'Correlación gaming_hours/reaction_time_ms: {corr_reaction:.4f}')
print(f'Correlación gaming_hours/addiction_score: {corr_addiction:.4f}')
print(f'addiction_score negativos en el original: {negativos_addiction}')
comparar('correlación gaming_hours/reaction_time_ms', float(corr_reaction), -0.94, tolerancia=0.01)
comparar('correlación gaming_hours/addiction_score', float(corr_addiction), 0.91, tolerancia=0.01)
comparar('addiction_score negativos', negativos_addiction, 107)
registrar('3.7', 'Descarte de columnas', 'reaction_time_ms', 'Correlación con gaming_hours', 0, f'r={corr_reaction:.4f}', 'Redundante con gaming_hours y fuera del objetivo definido.')
registrar('3.7', 'Descarte de columnas', 'addiction_score', 'Valores negativos y correlación con gaming_hours', negativos_addiction, f'r={corr_addiction:.4f}; negativos={negativos_addiction}', 'Escala no documentada; los negativos son anomalías del original y no se corrigen porque la columna se descarta.')

COLUMNAS_DESCARTADAS = {
    'student_id': 'Identificador único sin información analítica; coincide con número de fila + 1.',
    'reaction_time_ms': 'Redundante con gaming_hours y no vinculada al objetivo del proyecto.',
    'gaming_genre': 'Fuera del alcance: el análisis se centra en horas de juego, no tipo de juego.',
    'addiction_score': 'Escala no documentada, valores negativos y casi redundante con gaming_hours.'
}
for columna, justificacion in COLUMNAS_DESCARTADAS.items():
    registrar('3.7', 'Descarte de columnas', columna, 'Columna fuera del dataset analítico final', n0, 'Descartar', justificacion)
df = df.drop(columns=list(COLUMNAS_DESCARTADAS))

COLUMNAS_CONSERVADAS = ['age', 'gender', 'gaming_hours', 'study_hours', 'sleep_hours', 'attendance', 'social_activity', 'device_usage', 'stress_level', 'grades']
columnas_finales = ['age', 'gender', 'gaming_hours', 'study_hours', 'sleep_hours', 'attendance', 'social_activity', 'device_usage', 'uso_no_gaming', 'stress_level', 'gaming_cat', 'grades', 'grades_original', 'flag_grades_gt100', 'flag_ajustado']
df = df[columnas_finales]
print(f'Columnas descartadas: {list(COLUMNAS_DESCARTADAS)}')
print(f'Columnas conservadas explícitamente: {COLUMNAS_CONSERVADAS}')

### 3.8 Validaciones finales
Se comprueba con `assert` la forma, columnas, tipos, flags, rangos y ausencia de duplicados o nulos.

In [ ]:
columnas_finales = ['age', 'gender', 'gaming_hours', 'study_hours', 'sleep_hours', 'attendance', 'social_activity', 'device_usage', 'uso_no_gaming', 'stress_level', 'gaming_cat', 'grades', 'grades_original', 'flag_grades_gt100', 'flag_ajustado']
assert len(df) == n0
assert not df.isna().any().any()
assert df['grades'].between(0, 100).all()
assert (df['uso_no_gaming'] >= 0).all()
assert not df.duplicated().any()
assert df.columns.tolist() == columnas_finales
assert isinstance(df['stress_level'].dtype, pd.CategoricalDtype) and df['stress_level'].cat.ordered
assert isinstance(df['gaming_cat'].dtype, pd.CategoricalDtype) and df['gaming_cat'].cat.ordered
assert df['flag_grades_gt100'].dtype == bool
assert df['flag_ajustado'].dtype == bool
assert df_raw.shape == (8000, 14)
print('Validaciones finales: OK')

### 3.9 Exportación y resumen
Se guardan el dataset limpio y la bitácora, y se verifica que el CSV pueda recargarse con la misma estructura.

In [ ]:
bitacora = pd.DataFrame(registros_bitacora, columns=['n_paso', 'paso', 'columna', 'revision_o_problema', 'registros_afectados', 'decision', 'fundamento', 'filas_despues'])
df.to_csv(CLEAN_PATH, index=False, encoding='utf-8')
bitacora.to_csv(BITACORA_PATH, index=False, encoding='utf-8')

df_guardado = pd.read_csv(CLEAN_PATH)
assert df_guardado.shape == (n0, len(columnas_finales))
assert df_guardado.columns.tolist() == columnas_finales
print(f'Dataset guardado en: {CLEAN_PATH}')
print(f'Bitácora guardada en: {BITACORA_PATH}')
print(f'Filas: {n0} antes / {len(df)} después')
print(f'Columnas: {len(df_raw.columns)} antes / {len(df.columns)} después')
print(f'Columnas eliminadas: {list(COLUMNAS_DESCARTADAS)}')
print('Columnas creadas: uso_no_gaming, gaming_cat, grades_original, flag_grades_gt100, flag_ajustado')
print('Bitácora completa:')
display(bitacora)

#### Recarga posterior del dataset limpio
Las secciones 4 y 5 trabajarán directamente con el DataFrame `df` limpio que queda en memoria, sin recargarlo desde el CSV. El CSV se exporta como entregable y para reproducibilidad.

Si se necesita recargarlo en otra sesión, se deben restaurar las categorías ordenadas así:

```python
df = pd.read_csv('../data/processed/gaming_academic_clean.csv')
df['stress_level'] = pd.Categorical(df['stress_level'], categories=['Low', 'Medium', 'High'], ordered=True)
df['gaming_cat'] = pd.Categorical(df['gaming_cat'], categories=['Bajo', 'Medio', 'Alto'], ordered=True)
```

## Sección 4. Análisis univariado y bivariado

Esta sección describe la distribución individual de las variables y las relaciones entre ellas. Todos los cálculos se hacen sobre `df`, que ya fue limpiado en la sección 3; el dataset original y el archivo CSV no se modifican.

**Criterio de lectura:** la media resume bien una variable cuando la distribución es aproximadamente simétrica y no presenta valores extremos dominantes. Cuando hay asimetría o una diferencia importante entre media y mediana, se prioriza la mediana y el rango intercuartílico porque son medidas más robustas.

**Análisis temporal:** el dataset no contiene una fecha, hora ni otra variable temporal identificable. Por esa razón, no corresponde calcular media móvil, estacionalidad semanal/mensual ni quiebres temporales; esta ausencia se deja documentada en la conclusión de la sección.

### 4.0 Análisis univariado

El análisis univariado estudia una sola variable por vez. En las variables numéricas se resumen el centro, la dispersión, la forma y los valores extremos mediante tablas, histogramas y diagramas de caja. En las variables categóricas se cuentan las categorías y se calcula su proporción mediante frecuencias, porcentajes y gráficos de barras.

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Se excluyen los flags porque son indicadores booleanos, no variables numéricas de medición.
columnas_numericas = [
    columna for columna in df.select_dtypes(include=np.number).columns
    if columna not in ['flag_grades_gt100', 'flag_ajustado']
]

# Estos estadísticos describen centro, dispersión, forma y extremos de cada variable.
def moda_principal(serie):
    modas = serie.mode(dropna=True)
    return modas.iloc[0] if len(modas) else np.nan

def coeficiente_variacion(serie):
    media = serie.mean()
    return np.nan if np.isclose(media, 0) else serie.std(ddof=1) / abs(media) * 100

resumen_numerico = pd.DataFrame(index=columnas_numericas)
resumen_numerico['media'] = df[columnas_numericas].mean()
resumen_numerico['mediana'] = df[columnas_numericas].median()
resumen_numerico['moda'] = [moda_principal(df[columna]) for columna in columnas_numericas]
resumen_numerico['desvio_estandar'] = df[columnas_numericas].std()
resumen_numerico['coeficiente_variacion_%'] = [coeficiente_variacion(df[columna]) for columna in columnas_numericas]
resumen_numerico['minimo'] = df[columnas_numericas].min()
resumen_numerico['maximo'] = df[columnas_numericas].max()
resumen_numerico['rango_intercuartilico'] = df[columnas_numericas].quantile(0.75) - df[columnas_numericas].quantile(0.25)
for porcentaje in [5, 25, 50, 75, 95]:
    resumen_numerico[f'percentil_{porcentaje}'] = df[columnas_numericas].quantile(porcentaje / 100)
resumen_numerico['asimetria'] = df[columnas_numericas].skew()
resumen_numerico['curtosis_exceso'] = df[columnas_numericas].kurt()

print('Resumen univariado de variables numéricas:')
display(resumen_numerico.round(3))

# La diferencia entre media y mediana, junto con la asimetría, orienta qué medida de centro conviene reportar.
resumen_numerico['diferencia_media_mediana_%'] = (
    (resumen_numerico['media'] - resumen_numerico['mediana']).abs()
    / resumen_numerico['mediana'].replace(0, np.nan).abs() * 100
)
resumen_numerico['centro_recomendado'] = np.where(
    (resumen_numerico['asimetria'].abs() > 1)
    | (resumen_numerico['diferencia_media_mediana_%'] > 10),
    'Mediana e IQR',
    'Media y desvío estándar'
)
print('Criterio de centro recomendado: |asimetría| > 1 o diferencia media-mediana > 10% indica posible asimetría relevante.')
display(resumen_numerico[['media', 'mediana', 'asimetria', 'curtosis_exceso', 'diferencia_media_mediana_%', 'centro_recomendado']].round(3))

# Histogramas y cajas muestran forma, concentración, dispersión y posibles valores extremos.
filas = math.ceil(len(columnas_numericas) / 3)
fig, ejes = plt.subplots(filas, 3, figsize=(16, 4 * filas))
ejes = np.atleast_1d(ejes).ravel()
for eje, columna in zip(ejes, columnas_numericas):
    eje.hist(df[columna], bins=30, color='#2a9d8f', edgecolor='white', alpha=0.85)
    eje.set_title(f'Histograma: {columna}')
    eje.set_xlabel(columna)
    eje.set_ylabel('Frecuencia')
for eje in ejes[len(columnas_numericas):]:
    eje.set_visible(False)
plt.tight_layout()
plt.show()

fig, eje = plt.subplots(figsize=(14, 6))
eje.boxplot([df[columna].dropna() for columna in columnas_numericas], tick_labels=columnas_numericas, orientation='horizontal', patch_artist=True)
eje.set_title('Diagramas de caja de las variables numéricas')
eje.set_xlabel('Valor')
eje.grid(axis='x', alpha=0.25)
plt.tight_layout()
plt.show()

### 4.1 Análisis univariado de variables categóricas

Este bloque sigue siendo univariado porque analiza cada variable categórica por separado: `gender`, `stress_level` y `gaming_cat`. Las frecuencias absolutas indican cuántos registros pertenecen a cada categoría y las relativas muestran su peso sobre el total. La concentración del 20% toma las categorías más frecuentes que representan aproximadamente el 20% de las categorías disponibles y calcula qué proporción de observaciones acumulan. Esto permite detectar categorías dominantes.

In [ ]:
columnas_categoricas = ['gender', 'stress_level', 'gaming_cat']

# Se calcula una tabla por variable para conocer cantidad y porcentaje de cada categoría.
frecuencias_categoricas = {}
concentracion_20 = []
for columna in columnas_categoricas:
    frecuencias = df[columna].value_counts(dropna=False).rename('frecuencia').to_frame()
    frecuencias['porcentaje_%'] = frecuencias['frecuencia'] / len(df) * 100
    frecuencias_categoricas[columna] = frecuencias
    print(f'Frecuencias de {columna}:')
    display(frecuencias.round(2))

    # ceil evita redondear a cero cuando una variable tiene pocas categorías.
    cantidad_categorias_20 = max(1, math.ceil(df[columna].nunique() * 0.20))
    principales = frecuencias.head(cantidad_categorias_20)
    concentracion_20.append({
        'variable': columna,
        'categorias_totales': df[columna].nunique(),
        'categorias_tomadas_20%': cantidad_categorias_20,
        'categorias_principales': ', '.join(map(str, principales.index)),
        'porcentaje_observaciones_acumulado_%': principales['porcentaje_%'].sum()
    })

print('Concentración: participación de las categorías más frecuentes que representan el 20% de las categorías:')
display(pd.DataFrame(concentracion_20).round(2))

# Los gráficos de barras facilitan comparar rápidamente la frecuencia de cada categoría.
fig, ejes = plt.subplots(1, len(columnas_categoricas), figsize=(15, 4))
for eje, columna in zip(ejes, columnas_categoricas):
    frecuencias = frecuencias_categoricas[columna]
    eje.bar(frecuencias.index.astype(str), frecuencias['frecuencia'], color='#e76f51')
    eje.set_title(f'Frecuencias: {columna}')
    eje.set_xlabel('Categoría')
    eje.set_ylabel('Registros')
    eje.tick_params(axis='x', rotation=25)
plt.tight_layout()
plt.show()

### 4.2 Análisis bivariado: numérica frente a numérica

El análisis bivariado estudia dos variables al mismo tiempo para describir una relación, asociación o diferencia entre ellas. En este bloque se comparan dos variables numéricas mediante diagramas de dispersión y correlaciones. La correlación de Pearson mide asociación lineal y la de Spearman mide asociación monótona basada en rangos. Compararlas ayuda a distinguir una relación lineal de una relación que puede ser creciente/decreciente pero no lineal. La matriz de correlaciones resume todas las asociaciones numéricas; no implica causalidad.

In [ ]:
# Se calculan ambas correlaciones para evaluar relaciones lineales y monótonas entre variables.
correlacion_pearson = df[columnas_numericas].corr(method='pearson')
correlacion_spearman = df[columnas_numericas].corr(method='spearman')

print('Correlación de Pearson:')
display(correlacion_pearson.round(3))
print('Correlación de Spearman:')
display(correlacion_spearman.round(3))

# Estas parejas responden directamente al objetivo: juego/rendimiento y variables de contexto/rendimiento.
parejas_scatter = [
    ('gaming_hours', 'grades'),
    ('study_hours', 'grades'),
    ('sleep_hours', 'grades'),
    ('gaming_hours', 'study_hours')
]
fig, ejes = plt.subplots(2, 2, figsize=(13, 10))
for eje, (variable_x, variable_y) in zip(ejes.ravel(), parejas_scatter):
    eje.scatter(df[variable_x], df[variable_y], s=8, alpha=0.20, color='#264653')
    eje.set_title(f'{variable_y} frente a {variable_x}')
    eje.set_xlabel(variable_x)
    eje.set_ylabel(variable_y)
    eje.grid(alpha=0.2)
plt.tight_layout()
plt.show()

# El mapa de calor permite localizar rápidamente asociaciones positivas y negativas.
fig, eje = plt.subplots(figsize=(12, 9))
imagen = eje.imshow(correlacion_pearson, cmap='RdBu_r', vmin=-1, vmax=1)
eje.set_xticks(range(len(columnas_numericas)), columnas_numericas, rotation=45, ha='right')
eje.set_yticks(range(len(columnas_numericas)), columnas_numericas)
for fila in range(len(columnas_numericas)):
    for columna in range(len(columnas_numericas)):
        eje.text(columna, fila, f'{correlacion_pearson.iloc[fila, columna]:.2f}', ha='center', va='center', fontsize=8)
fig.colorbar(imagen, ax=eje, label='Correlación de Pearson')
eje.set_title('Mapa de calor de correlaciones de Pearson')
plt.tight_layout()
plt.show()

# Se listan las asociaciones más fuertes para facilitar la interpretación sin contar la diagonal.
pares_correlacion = []
for indice, variable_x in enumerate(columnas_numericas):
    for variable_y in columnas_numericas[indice + 1:]:
        pares_correlacion.append({
            'variable_1': variable_x,
            'variable_2': variable_y,
            'pearson': correlacion_pearson.loc[variable_x, variable_y],
            'spearman': correlacion_spearman.loc[variable_x, variable_y]
        })
print('Pares con mayor asociación absoluta según Pearson:')
display(pd.DataFrame(pares_correlacion).sort_values('pearson', key=lambda serie: serie.abs(), ascending=False).head(10).round(3))

### 4.3 Análisis bivariado: numérica frente a categórica

Este bloque es bivariado porque relaciona dos variables: `grades` (numérica) y cada variable de grupo (`gender`, `stress_level` y `gaming_cat`) por separado. Se compara `grades` entre grupos. La tabla resume el centro y la dispersión por categoría; los diagramas de caja y violín muestran diferencias de distribución, solapamiento y posibles valores extremos. La comparación es descriptiva y no prueba causalidad.

In [ ]:
columnas_grupo = ['gender', 'stress_level', 'gaming_cat']

# Estos estadísticos permiten comparar rendimiento típico, variabilidad y rango dentro de cada grupo.
resumen_grupos = {}
for columna in columnas_grupo:
    resumen = df.groupby(columna, observed=True)['grades'].agg(
        cantidad='count',
        media='mean',
        mediana='median',
        desvio_estandar='std',
        minimo='min',
        maximo='max'
    )
    resumen['rango_intercuartilico'] = df.groupby(columna, observed=True)['grades'].quantile(0.75) - df.groupby(columna, observed=True)['grades'].quantile(0.25)
    resumen_grupos[columna] = resumen
    print(f'Estadísticos de grades por {columna}:')
    display(resumen.round(3))

# Cajas y violines muestran la distribución completa, no solo promedios por grupo.
fig, ejes = plt.subplots(1, len(columnas_grupo), figsize=(16, 5))
for eje, columna in zip(ejes, columnas_grupo):
    categorias = list(df[columna].dropna().unique())
    categorias = [categoria for categoria in df[columna].cat.categories if categoria in categorias] if hasattr(df[columna], 'cat') else categorias
    datos = [df.loc[df[columna] == categoria, 'grades'].dropna().to_numpy() for categoria in categorias]
    posiciones = np.arange(1, len(categorias) + 1)
    eje.violinplot(datos, positions=posiciones, showmeans=True, showmedians=True)
    eje.boxplot(datos, positions=posiciones, widths=0.12, patch_artist=True, boxprops={'facecolor': '#f4a261', 'alpha': 0.8})
    eje.set_xticks(posiciones, [str(categoria) for categoria in categorias], rotation=25)
    eje.set_title(f'grades por {columna}')
    eje.set_xlabel('Categoría')
    eje.set_ylabel('grades')
    eje.grid(axis='y', alpha=0.2)
plt.tight_layout()
plt.show()

### 4.4 Análisis bivariado: categórica frente a categórica

Este bloque es bivariado porque estudia la relación entre dos variables categóricas a la vez: `gender` frente a `stress_level`, `gender` frente a `gaming_cat` y `stress_level` frente a `gaming_cat`. Las tablas de contingencia muestran frecuencias absolutas, porcentajes condicionados por fila y porcentajes condicionados por columna.

In [ ]:
# Cada tabla permite leer una perspectiva distinta de la asociación entre dos variables categóricas.
for indice, variable_filas in enumerate(columnas_categoricas):
    for variable_columnas in columnas_categoricas[indice + 1:]:
        tabla_absoluta = pd.crosstab(df[variable_filas], df[variable_columnas], dropna=False)
        tabla_por_fila = pd.crosstab(df[variable_filas], df[variable_columnas], normalize='index', dropna=False) * 100
        tabla_por_columna = pd.crosstab(df[variable_filas], df[variable_columnas], normalize='columns', dropna=False) * 100

        print(f'{variable_filas} frente a {variable_columnas}: frecuencias absolutas')
        display(tabla_absoluta)
        print(f'{variable_filas} frente a {variable_columnas}: porcentajes por fila')
        display(tabla_por_fila.round(2))
        print(f'{variable_filas} frente a {variable_columnas}: porcentajes por columna')
        display(tabla_por_columna.round(2))

### 4.5 Lectura integrada de resultados

- **Forma de las distribuciones:** las variables numéricas presentan asimetrías cercanas a cero; por eso, en general, la media y el desvío estándar son resúmenes razonables. `grades` tiene asimetría negativa moderada (`-0,283`) y un máximo acumulado en 100 por el tope aplicado, por lo que conviene acompañar la media con mediana e IQR. `grades_original` se conserva como referencia, pero no debe analizarse junto con `grades` en un mismo modelo porque son casi la misma variable.
- **Asociaciones numéricas:** `study_hours` muestra una asociación positiva importante con `grades` (Pearson ≈ 0,734; Spearman ≈ 0,742), mientras `gaming_hours` muestra una asociación negativa moderada (Pearson ≈ -0,558; Spearman ≈ -0,547). `sleep_hours` tiene una asociación positiva más débil (Pearson ≈ 0,245). Son asociaciones descriptivas y no demuestran causalidad.
- **Comparaciones por grupo:** el promedio de `grades` aumenta desde `gaming_cat=Alto` (≈ 51,40) a `Medio` (≈ 66,87) y `Bajo` (≈ 79,92). Por `stress_level`, el promedio es ≈ 50,50 en `Low`, ≈ 72,62 en `Medium` y ≈ 80,89 en `High`; este patrón debe interpretarse con cautela porque puede reflejar la forma en que se generó el dataset simulado y no una relación causal real.
- **Categorías:** `gender` está concentrado en `Male` (48,80%) y `Female` (47,54%), mientras `gaming_cat` está balanceada en aproximadamente un tercio por grupo. La tabla de contingencia revela una asociación estructural fuerte entre `stress_level` y `gaming_cat`: `Low` se concentra en `Alto` y `Medium` en `Bajo`, por lo que estas variables no deben interpretarse como independientes.
- **Tiempo:** no hay una variable de fecha u hora en el dataset. Por lo tanto, no se calculan media móvil, estacionalidad semanal/mensual ni quiebres temporales; esos análisis no son aplicables a esta fase.

## Sección 5. Análisis descriptivo y exploratorio

El análisis exploratorio integra los resultados anteriores para describir el fenómeno completo: cómo se distribuyen las horas de juego, estudio y sueño; cómo se relacionan con el rendimiento; y qué diferencias aparecen entre grupos. Cada visualización responde una pregunta concreta y tiene su interpretación inmediatamente debajo.

**Relación con la Sección 4:** es esperable que algunos gráficos aparezcan en ambas secciones. En la Sección 4 se presentan como análisis univariado o bivariado específico, para medir una variable o una relación entre dos variables. En esta Sección 5 se reutilizan o combinan como evidencia para construir una caracterización integrada, identificar patrones y anomalías, y formular hipótesis para la Fase 2; por eso la interpretación aquí se enfoca en el fenómeno completo y no repite solamente los estadísticos.

**Nota sobre el tiempo:** el dataset no contiene fecha, hora ni período de observación. Por lo tanto, no es válido analizar estacionalidad semanal/mensual ni quiebres temporales. Se incluye una serie por orden de registro con media móvil únicamente para revisar si existe un patrón de ordenamiento en los datos; no debe interpretarse como evolución temporal.

### 5.1 Distribución del rendimiento

Este histograma permite observar la concentración de las calificaciones, su forma general y si aparecen acumulaciones en determinados rangos. La línea vertical marca la media y ayuda a compararla visualmente con la distribución.

In [ ]:
fig, eje = plt.subplots(figsize=(9, 5))
eje.hist(df['grades'], bins=25, color='#2a9d8f', edgecolor='white', alpha=0.85, label='Frecuencia de grades')
eje.axvline(df['grades'].mean(), color='#e76f51', linestyle='--', linewidth=2, label=f"Media = {df['grades'].mean():.2f}")
eje.set_title('Distribución de calificaciones académicas')
eje.set_xlabel('Calificación (escala 0-100)')
eje.set_ylabel('Cantidad de estudiantes')
eje.legend()
eje.grid(axis='y', alpha=0.2)
plt.tight_layout()
plt.show()

**Interpretación:** Las calificaciones se concentran principalmente en la zona media de la escala, con una media cercana a 66 puntos. La acumulación de valores en 100 refleja el tope aplicado durante la limpieza y debe considerarse al interpretar la forma de la distribución.

### 5.2 Rendimiento según nivel de juego

Este diagrama de caja compara la distribución de `grades` entre los terciles de `gaming_hours`. Permite observar diferencias de mediana, dispersión y valores extremos entre estudiantes con bajo, medio y alto tiempo de juego.

In [ ]:
categorias_gaming = ['Bajo', 'Medio', 'Alto']
datos_gaming = [df.loc[df['gaming_cat'] == categoria, 'grades'].dropna() for categoria in categorias_gaming]
fig, eje = plt.subplots(figsize=(9, 5))
eje.boxplot(datos_gaming, tick_labels=categorias_gaming, patch_artist=True, boxprops={'facecolor': '#f4a261', 'alpha': 0.8})
eje.set_title('Distribución de calificaciones según nivel de horas de juego')
eje.set_xlabel('Nivel de gaming_hours (terciles)')
eje.set_ylabel('Calificación (escala 0-100)')
eje.grid(axis='y', alpha=0.2)
plt.tight_layout()
plt.show()

**Interpretación:** La mediana de `grades` disminuye claramente desde el grupo Bajo al grupo Alto de `gaming_hours`. El patrón es consistente con la correlación negativa observada, aunque por sí solo no demuestra que jugar más cause menor rendimiento.

### 5.3 Relación entre horas de juego y calificación

Este diagrama de dispersión permite observar la dirección, intensidad y dispersión de la relación entre `gaming_hours` y `grades`. La recta superpuesta es una referencia descriptiva de tendencia lineal, no un modelo causal.

In [ ]:
fig, eje = plt.subplots(figsize=(9, 5))
eje.scatter(df['gaming_hours'], df['grades'], s=10, alpha=0.22, color='#264653', label='Estudiantes')
pendiente, intercepto = np.polyfit(df['gaming_hours'], df['grades'], 1)
valores_x = np.linspace(df['gaming_hours'].min(), df['gaming_hours'].max(), 100)
eje.plot(valores_x, pendiente * valores_x + intercepto, color='#e76f51', linewidth=2, label=f'Tendencia lineal (r = {df["gaming_hours"].corr(df["grades"]):.2f})')
eje.set_title('Horas de videojuegos y rendimiento académico')
eje.set_xlabel('Horas de videojuegos por día')
eje.set_ylabel('Calificación (escala 0-100)')
eje.legend()
eje.grid(alpha=0.2)
plt.tight_layout()
plt.show()

**Interpretación:** La nube presenta una tendencia descendente: a mayores horas de videojuegos, las calificaciones tienden a ser menores. La dispersión alrededor de la recta indica que `gaming_hours` no explica por sí sola todo el rendimiento.

### 5.4 Comparación de calificaciones por grupos

Este gráfico de barras compara la calificación media entre los niveles de `gaming_cat` y `stress_level`. Sirve para resumir diferencias de grupo de manera directa, manteniendo la misma escala de calificación para que las alturas sean comparables.

In [ ]:
media_gaming = df.groupby('gaming_cat', observed=True)['grades'].mean().reindex(['Bajo', 'Medio', 'Alto'])
media_stress = df.groupby('stress_level', observed=True)['grades'].mean().reindex(['Low', 'Medium', 'High'])
fig, ejes = plt.subplots(1, 2, figsize=(13, 5))
ejes[0].bar(media_gaming.index, media_gaming.values, color='#2a9d8f')
ejes[0].set_title('Calificación media según nivel de juego')
ejes[0].set_xlabel('Nivel de gaming_hours')
ejes[0].set_ylabel('Calificación media (escala 0-100)')
ejes[0].set_ylim(0, 100)
ejes[0].grid(axis='y', alpha=0.2)
ejes[1].bar(media_stress.index, media_stress.values, color='#e9c46a')
ejes[1].set_title('Calificación media según nivel de estrés')
ejes[1].set_xlabel('Nivel de stress_level')
ejes[1].set_ylabel('Calificación media (escala 0-100)')
ejes[1].set_ylim(0, 100)
ejes[1].grid(axis='y', alpha=0.2)
plt.tight_layout()
plt.show()

**Interpretación:** La media de `grades` es mayor en el grupo Bajo de juego y disminuye hacia el grupo Alto. También se observan diferencias importantes entre niveles de estrés; como este patrón es contrario a una interpretación intuitiva, debe contrastarse en la Fase 2 y revisarse junto con la posible estructura del dataset simulado.

### 5.5 Correlaciones entre variables numéricas

Este mapa de calor permite revisar simultáneamente la dirección e intensidad de las asociaciones entre las variables numéricas. Se excluyen los indicadores booleanos y se conserva una escala común de -1 a 1 para no exagerar diferencias.

In [ ]:
correlaciones_exploratorias = df[columnas_numericas].corr(method='pearson')
fig, eje = plt.subplots(figsize=(11, 8))
imagen = eje.imshow(correlaciones_exploratorias, cmap='RdBu_r', vmin=-1, vmax=1)
eje.set_xticks(range(len(columnas_numericas)), columnas_numericas, rotation=45, ha='right')
eje.set_yticks(range(len(columnas_numericas)), columnas_numericas)
for fila in range(len(columnas_numericas)):
    for columna in range(len(columnas_numericas)):
        eje.text(columna, fila, f'{correlaciones_exploratorias.iloc[fila, columna]:.2f}', ha='center', va='center', fontsize=8)
fig.colorbar(imagen, ax=eje, label='Correlación de Pearson')
eje.set_title('Mapa de calor de correlaciones numéricas')
plt.tight_layout()
plt.show()

**Interpretación:** Las asociaciones más visibles con `grades` son positiva con `study_hours` y negativa con `gaming_hours` y `device_usage`. La correlación alta entre `grades` y `grades_original` es esperable porque ambas representan prácticamente la misma medición, por lo que no deben incluirse juntas en un modelo.

### 5.6 Composición de niveles de estrés dentro de cada nivel de juego

Este gráfico de barras apiladas muestra cómo se distribuyen los niveles de `stress_level` dentro de cada categoría de `gaming_cat`. Los porcentajes permiten comparar composiciones aunque los grupos no tuvieran exactamente el mismo tamaño.

In [ ]:
tabla_composicion = pd.crosstab(df['gaming_cat'], df['stress_level'], normalize='index').reindex(index=['Bajo', 'Medio', 'Alto'], columns=['Low', 'Medium', 'High']).fillna(0) * 100
fig, eje = plt.subplots(figsize=(9, 5))
base = np.zeros(len(tabla_composicion))
colores = {'Low': '#457b9d', 'Medium': '#e9c46a', 'High': '#e76f51'}
for nivel_estres in tabla_composicion.columns:
    eje.bar(tabla_composicion.index, tabla_composicion[nivel_estres], bottom=base, label=nivel_estres, color=colores[nivel_estres])
    base += tabla_composicion[nivel_estres].to_numpy()
eje.set_title('Composición porcentual de estrés según nivel de juego')
eje.set_xlabel('Nivel de gaming_hours (terciles)')
eje.set_ylabel('Porcentaje de estudiantes (%)')
eje.set_ylim(0, 100)
eje.legend(title='Nivel de estrés')
eje.grid(axis='y', alpha=0.2)
plt.tight_layout()
plt.show()

**Interpretación:** La composición no es similar entre los niveles de juego: `gaming_cat=Bajo` se concentra en `stress_level=Medium`, mientras `gaming_cat=Alto` se concentra en `Low`. Esta estructura conjunta es una posible anomalía del dataset simulado y puede afectar la interpretación de las diferencias de rendimiento.

### 5.7 Horas de estudio y rendimiento por nivel de juego

Este diagrama de dispersión examina si la relación entre `study_hours` y `grades` cambia según el nivel de `gaming_cat`. El color permite explorar visualmente posibles diferencias de grupo sin convertirlas todavía en una conclusión formal.

In [ ]:
fig, eje = plt.subplots(figsize=(9, 5))
colores_gaming = {'Bajo': '#2a9d8f', 'Medio': '#e9c46a', 'Alto': '#e76f51'}
for nivel, datos in df.groupby('gaming_cat', observed=True):
    eje.scatter(datos['study_hours'], datos['grades'], s=10, alpha=0.2, label=nivel, color=colores_gaming[str(nivel)])
eje.set_title('Relación entre horas de estudio y calificación según nivel de juego')
eje.set_xlabel('Horas de estudio por día')
eje.set_ylabel('Calificación (escala 0-100)')
eje.legend(title='Nivel de gaming_hours')
eje.grid(alpha=0.2)
plt.tight_layout()
plt.show()

**Interpretación:** En los tres niveles de juego se observa una tendencia positiva entre horas de estudio y calificaciones. La comparación por color permite evaluar en la Fase 2 si el rendimiento asociado al estudio se mantiene después de controlar formalmente por `gaming_hours` y otras variables.

### 5.8 Serie por orden de registro y media móvil

No existe una variable temporal en el dataset. Esta serie usa el orden de las filas como eje de observación y una media móvil para comprobar si hay bloques o cambios de composición en el archivo; no representa semanas, meses ni evolución temporal real.

In [ ]:
orden = np.arange(1, len(df) + 1)
media_movil = df['grades'].rolling(window=200, min_periods=1).mean()
fig, eje = plt.subplots(figsize=(12, 5))
eje.plot(orden, df['grades'], color='#264653', alpha=0.08, linewidth=0.6, label='Calificación individual')
eje.plot(orden, media_movil, color='#e76f51', linewidth=2, label='Media móvil de 200 registros')
eje.axhline(df['grades'].mean(), color='#2a9d8f', linestyle='--', linewidth=1.5, label=f"Media global = {df['grades'].mean():.2f}")
eje.set_title('Calificaciones según orden de registro: control de media móvil')
eje.set_xlabel('Orden de registro (sin unidad temporal)')
eje.set_ylabel('Calificación (escala 0-100)')
eje.set_ylim(0, 100)
eje.legend()
eje.grid(alpha=0.2)
plt.tight_layout()
plt.show()

**Interpretación:** La media móvil permite detectar cambios por bloques en el orden de almacenamiento. Como el eje no representa tiempo, cualquier variación debe interpretarse como posible ordenamiento o estructura del dataset, no como tendencia temporal, estacionalidad semanal/mensual o quiebre histórico.

### 5.9 Patrones y anomalías relevantes

1. **Relación negativa entre juego y rendimiento:** `gaming_hours` presenta una correlación de Pearson aproximada de `-0,558` con `grades`, y las medias descienden de aproximadamente `79,92` en `gaming_cat=Bajo` a `51,40` en `gaming_cat=Alto`. La evidencia aparece en la matriz de correlaciones, el diagrama de dispersión y el diagrama de cajas.
2. **Relación positiva entre estudio y rendimiento:** `study_hours` presenta una correlación de Pearson aproximada de `0,734` con `grades`. La nube de puntos mantiene una tendencia ascendente incluso al colorear por nivel de juego, lo que indica que el estudio es una variable importante para analizar conjuntamente en la siguiente fase.
3. **Estructura categórica poco natural entre estrés y juego:** `stress_level=Low` se concentra en `gaming_cat=Alto`, mientras `stress_level=Medium` se concentra en `gaming_cat=Bajo`; incluso aparecen combinaciones con frecuencia cero. La tabla de contingencia y el gráfico apilado sugieren una posible regla de generación o dependencia artificial del dataset simulado.
4. **Acumulación de calificaciones en el límite superior:** `grades` tiene numerosos registros en 100 y originalmente había valores superiores a 100 que fueron limitados al rango 0-100. Esto produce un efecto de techo y puede reducir la capacidad de distinguir a los estudiantes con mejor rendimiento.
5. **Ausencia de dimensión temporal:** el dataset no contiene fecha ni hora. La media móvil por orden de registro sirve únicamente para detectar ordenamiento interno, pero no permite concluir tendencias, estacionalidad ni quiebres temporales.

### 5.10 Hipótesis preliminares para la Fase 2

Estas hipótesis se formulan a partir de los patrones exploratorios y deberán contrastarse con métodos estadísticos o modelos en la Fase 2:

- **H1:** A mayor cantidad de horas de videojuegos por día, menor será la calificación académica esperada, controlando por horas de estudio, sueño y asistencia.
- **H2:** A mayor cantidad de horas de estudio por día, mayor será la calificación académica esperada, controlando por horas de videojuegos y variables de contexto.
- **H3:** La relación entre horas de videojuegos y calificación cambia según el nivel de estrés; existe una posible interacción entre `gaming_hours` y `stress_level`.
- **H4:** Las horas de sueño y la asistencia presentan una asociación positiva con el rendimiento académico, aunque de menor magnitud que las horas de estudio.

Las hipótesis expresan asociaciones a contrastar y no afirmaciones causales. Debido a que el dataset es simulado y no contiene información temporal, los resultados deberán interpretarse con cautela.

## Sección 6. Conclusión de la Fase 1
PENDIENTE.